<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-14-two-multimodal-helpers-for-fernwood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 14 (graded) — Two multimodal helpers for Fernwood
**Course 2: Generative AI and LLMs with Python — Chapter 14: Multimodal & diffusion in practice**

**Problem brief (Sam Okafor, Fernwood Media):** "Two asks: a concept-art helper for the
design desk, and automatic transcripts + summaries for our podcast."

**What you'll submit:** the concept-art helper (text-to-image + inpainting), the podcast
pipeline (transcription + summary), one VLM captioning call, and a build-vs-buy
recommendation for each.

In [ ]:
!pip install -q diffusers transformers accelerate soundfile datasets

## 1. Concept-art helper: text-to-image + inpainting

In [ ]:
import torch
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def generate_concept_art(prompt, negative_prompt='blurry, watermark, text', guidance_scale=7.5):
    try:
        from diffusers import AutoPipelineForText2Image
        pipe = AutoPipelineForText2Image.from_pretrained(
            'stabilityai/sd-turbo', torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        ).to(device)
        # sd-turbo is a distilled, few-step model — fast enough for a free Colab GPU
        image = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
        print('Generated with stabilityai/sd-turbo.')
        return image
    except Exception as e:
        print(f'Diffusion pipeline unavailable ({e}) — offline placeholder engaged.')
        from PIL import Image, ImageDraw
        img = Image.new('RGB', (256, 256), color=(60, 90, 120))
        d = ImageDraw.Draw(img)
        d.text((10, 120), f'[placeholder for:\n{prompt[:40]}]', fill=(255, 255, 255))
        return img

art = generate_concept_art('a moody watercolor mood board for a tech-noir news article, blue tones')
plt.imshow(art); plt.axis('off'); plt.title('Concept art'); plt.show()

In [ ]:
def inpaint_region(base_image, mask_box, prompt):
    try:
        from diffusers import AutoPipelineForInpainting
        from PIL import Image, ImageDraw
        mask = Image.new('L', base_image.size, 0)
        ImageDraw.Draw(mask).rectangle(mask_box, fill=255)
        pipe = AutoPipelineForInpainting.from_pretrained(
            'diffusers/stable-diffusion-xl-1.0-inpainting-0.1' if device == 'cuda' else 'runwayml/stable-diffusion-inpainting',
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        ).to(device)
        result = pipe(prompt=prompt, image=base_image.resize((512, 512)), mask_image=mask.resize((512, 512)),
                       num_inference_steps=15).images[0]
        print('Inpainted with a real diffusers pipeline.')
        return result
    except Exception as e:
        print(f'Inpainting pipeline unavailable ({e}) — returning the original image unmodified.')
        return base_image

inpainted = inpaint_region(art, (60, 60, 180, 180), 'a glowing neon sign')
plt.imshow(inpainted); plt.axis('off'); plt.title('Inpainted region'); plt.show()

## 2. Podcast pipeline: Whisper transcription + LLM summary

In [ ]:
def load_sample_audio():
    try:
        from datasets import load_dataset
        ds = load_dataset('hf-internal-testing/librispeech_asr_dummy', 'clean', split='validation')
        sample = ds[0]['audio']
        print('Loaded a real short speech sample from LibriSpeech (dummy split).')
        return sample['array'], sample['sampling_rate']
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — transcription step will use placeholder text.')
        return None, None

audio_array, sr = load_sample_audio()

from transformers import pipeline

if audio_array is not None:
    asr = pipeline('automatic-speech-recognition', model='openai/whisper-tiny', device=0 if device == 'cuda' else -1)
    transcript = asr({'array': audio_array, 'sampling_rate': sr})['text']
else:
    transcript = ('Welcome back to the show. Today we are discussing the future of renewable '
                  'energy storage and what it means for the grid over the next decade.')

print('Transcript:', transcript)

In [ ]:
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-6-6', device=0 if device == 'cuda' else -1)
summary_input = transcript if len(transcript.split()) > 30 else transcript * 5  # tiny samples need padding for the model
summary = summarizer(summary_input, max_length=40, min_length=10, do_sample=False)[0]['summary_text']
print('Summary:', summary)

## 3. One VLM captioning call

In [ ]:
captioner = pipeline('image-to-text', model='Salesforce/blip-image-captioning-base', device=0 if device == 'cuda' else -1)
caption = captioner(art)[0]['generated_text']
print('VLM caption of the concept-art image:', caption)

## 4. Build vs. buy (fill in)
For the concept-art helper and the podcast pipeline separately: which would you recommend
Fernwood build in-house (like this notebook does) vs. buy from a vendor, given the compute
cost you just felt running these models? What's different about the two tasks that changes
the answer?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 14: Multimodal & diffusion in practice*